
# Module 2 — Ungraded Lab (LO2)
**Design the Experiment for a Personalization Launch**  
> **Solution Notebook** — one worked approach for each PRACTICE CHALLENGE.


In [ ]:

# ================================================
# Setup & Synthetic Data
# ================================================
import numpy as np, pandas as pd

rng = np.random.default_rng(42)

# Config
N_USERS = 120_000
TX_PER_USER_MEAN = 1.8
REGIONS = ["NA","EU","LATAM","APAC"]
region_probs = [0.35, 0.25, 0.25, 0.15]

# Users
users = pd.DataFrame({
    "user_id": np.arange(N_USERS),
    "region": rng.choice(REGIONS, size=N_USERS, p=region_probs),
    "device_id": rng.integers(10_000, 99_999, N_USERS)
})

# Transactions per user (Poisson)
tx_counts = rng.poisson(lam=TX_PER_USER_MEAN, size=N_USERS)
rows = []
start = pd.Timestamp("2025-02-01")

for uid, reg, dev, k in zip(users.user_id, users.region, users.device_id, tx_counts):
    if k == 0: 
        continue
    ts = start + pd.to_timedelta(rng.integers(0, 14, k), unit="D")  # 2-week exposure window baseline
    # regional base fraud risk
    base = {"NA":0.007, "EU":0.0045, "LATAM":0.011, "APAC":0.0065}[reg]
    # treatment effectiveness (loss avoidance) if assigned to treatment and model flags fraud
    for t in ts:
        rows.append((uid, reg, dev, t, base))

df = pd.DataFrame(rows, columns=["user_id","region","device_id","event_ts","base_fraud_rate"])
N = len(df)

# Assignment at user-level by default (you will choose design later)
df["assigned_variation"] = rng.choice(["control","treatment"], size=N)

# True fraud outcome
is_fraud_true = rng.random(N) < df["base_fraud_rate"].values
df["is_fraud_true"] = is_fraud_true

# Fraud amount (skewed)
fraud_loss = np.where(is_fraud_true, rng.lognormal(mean=3.2, sigma=0.9, size=N), 0.0)
df["fraud_loss"] = fraud_loss

# Model action probability (only for treatment arm)
model_flags = (df["assigned_variation"]=="treatment") & (rng.random(N) < 0.35)
df["model_flag"] = model_flags

# If model flags and fraud true, some losses are avoided (e.g., decline or step-up)
avoid_ratio = 0.35  # average proportion avoided when correctly flagged
df["loss_avoided"] = np.where(df["model_flag"] & df["is_fraud_true"], df["fraud_loss"] * avoid_ratio, 0.0)

# Label delays (days)
label_delay_days = rng.lognormal(mean=2.0, sigma=0.8, size=N).astype(int)  # right-skewed
df["label_available_ts"] = df["event_ts"] + pd.to_timedelta(label_delay_days, unit="D")

# Operational signals / guardrails
df["approved"] = ~(df["model_flag"] & rng.random(N)<0.25)  # some flagged tx get declined
df["approval_latency_ms"] = np.where(df["model_flag"], rng.normal(420, 80, N).clip(120, 900), rng.normal(300, 60, N).clip(120, 700))
df["support_tickets"] = rng.poisson(lam=np.where(df["approved"], 0.002, 0.006), size=N)

print("Transactions:", len(df), "Users:", N_USERS)
df.head()


In [ ]:

# ================================================
# Helper Functions
# ================================================
import numpy as np
import pandas as pd
from math import ceil
from scipy.stats import norm

def window_labels(df, exposure_days=14, outcome_days=30, anchor=None):
    """Filter events by exposure window and compute which labels are observable within outcome window."""
    d = df.copy()
    if anchor is None:
        anchor = pd.Timestamp("2025-02-01")
    exposure_end = anchor + pd.to_timedelta(exposure_days, unit="D")
    outcome_cutoff = exposure_end + pd.to_timedelta(outcome_days, unit="D")
    d = d[(d.event_ts >= anchor) & (d.event_ts < exposure_end)].copy()
    d["label_observed"] = d["label_available_ts"] <= outcome_cutoff
    d["observed_fraud"] = d["is_fraud_true"] & d["label_observed"]
    d["observed_loss"] = np.where(d["label_observed"], d["fraud_loss"], 0.0)
    d["observed_loss_avoided"] = np.where(d["label_observed"], d["loss_avoided"], 0.0)
    return d, {"anchor": anchor, "exposure_end": exposure_end, "outcome_cutoff": outcome_cutoff}

def estimate_spillover(df):
    """Crude spillover score: share of devices appearing in both arms + cross-region share."""
    # Device cross-arm
    pivot = df.groupby(["device_id","assigned_variation"]).size().unstack(fill_value=0)
    cross_arm_devices = (pivot.gt(0).sum(axis=1) > 1).mean()
    # Cross-region traffic per device
    dev_regs = df.groupby("device_id")["region"].nunique().mean()
    return {
        "cross_arm_device_share": float(cross_arm_devices),
        "avg_regions_per_device": float(dev_regs),
        "spillover_risk_score": float(0.5*cross_arm_devices + 0.5*min(dev_regs/3,1.0))
    }

def design_recommendation(spillover_score, region_var_coef=0.15, label_delay_days=30):
    """Heuristic recommendation."""
    if spillover_score > 0.25 or region_var_coef > 0.2:
        return "geo_experiment"
    if label_delay_days > 30:
        return "stepped_wedge"
    return "user_ab"

# Power/sample size (two-proportion z-test approximation for fraud-loss rate per unit)
def n_per_arm_for_proportions(p_ctrl, mde_abs, alpha=0.05, power=0.8):
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    p_treat = p_ctrl - mde_abs
    var = p_ctrl*(1-p_ctrl) + p_treat*(1-p_treat)
    n = var * (z_alpha + z_beta)**2 / (mde_abs**2)
    return ceil(n)

def design_effect(cluster_size, icc):
    return 1 + (cluster_size - 1) * icc

def cuped_adjust(y_post, y_pre):
    """Return adjusted post metric and theta; simple regression-adjusted CUPED."""
    cov = np.cov(y_post, y_pre, ddof=1)
    theta = cov[0,1] / cov[1,1] if cov[1,1] != 0 else 0.0
    y_adj = y_post - theta * y_pre
    return y_adj, theta

def guardrail_report(df_obs, thresholds=None):
    if thresholds is None:
        thresholds = {
            "max_fpr": 0.015,
            "max_latency_ms": 500,
            "tickets_diff_max": 0.0005,
            "chargeback_rate_max": 0.006
        }
    d = df_obs.copy()
    treat = d[d.assigned_variation=="treatment"]
    ctrl  = d[d.assigned_variation=="control"]
    # False positive rate proxy: flagged but not fraud (observed)
    fp_treat = ((treat.model_flag) & (~treat.observed_fraud)).mean()
    fp_ctrl  = ((ctrl.model_flag) & (~ctrl.observed_fraud)).mean()
    fpr = fp_treat - fp_ctrl
    latency_diff = treat.approval_latency_ms.mean() - ctrl.approval_latency_ms.mean()
    tickets_diff = treat.support_tickets.mean() - ctrl.support_tickets.mean()
    # Chargeback rate proxy (observed)
    chb_treat = (treat.observed_fraud & treat.approved).mean()
    checks = {
        "FPR within limit": fpr <= thresholds["max_fpr"],
        "Latency within limit": latency_diff <= thresholds["max_latency_ms"],
        "Support tickets not higher": tickets_diff <= thresholds["tickets_diff_max"],
        "Chargeback rate within limit": chb_treat <= thresholds["chargeback_rate_max"]
    }
    decision = "GO" if all(checks.values()) else "NO-GO"
    return {
        "fpr_excess": float(fpr),
        "latency_diff_ms": float(latency_diff),
        "tickets_diff": float(tickets_diff),
        "chargeback_rate_treat": float(chb_treat),
        "checks": checks,
        "decision": decision
    }

print("Helpers ready.")



## Activity 1 — Unit of Assignment & Design Choice
Inspect feasibility diagnostics and select unit + design. Justify briefly.


In [ ]:

### SOLUTION — Activity 1
spill = estimate_spillover(df)
# Choose unit/design based on heuristics
if spill["spillover_risk_score"] > 0.25:
    unit_of_assignment = "geo"
    design_choice = "geo_experiment"
else:
    unit_of_assignment = "user"
    design_choice = "user_ab"
rec = design_recommendation(spill["spillover_risk_score"], region_var_coef=0.18, label_delay_days=30)
{"spill": spill, "unit": unit_of_assignment, "design": design_choice, "heuristic_recommendation": rec}



**Rationale (example):**  
- **Interference risk** is moderate; we start with **user A/B** for speed, keeping a **geo back-up** if cross-arm devices spike.  
- **Constraints:** label delays manageable with a 30‑day outcome window; region variance tracked with stratified analysis.



## Activity 2 — Exposure & Outcome Windows
Compare label coverage for different windows and select your final choice.


In [ ]:

### SOLUTION — Activity 2
coverage_rows = []
for wd in [14, 30, 45]:
    d_obs, meta = window_labels(df, exposure_days=14, outcome_days=wd)
    coverage = d_obs["label_observed"].mean()
    coverage_rows.append({"outcome_days": wd, "label_coverage": float(coverage)})
pd.DataFrame(coverage_rows)



**Decision:** exposure window 14 days; outcome window **30 days** (good coverage vs timeliness).  
- 14d too low (under‑measures fraud).  
- 45d higher coverage but slows decisions and risks seasonality shifts.



## Activity 3 — Spillover Mitigation
Quantify spillover and apply one mitigation. Re-estimate effective sample size.


In [ ]:

### SOLUTION — Activity 3
spill_before = estimate_spillover(df)
multi_arm_devices = df.groupby(["device_id","assigned_variation"]).size().unstack(fill_value=0)
to_exclude = multi_arm_devices.index[multi_arm_devices.gt(0).sum(axis=1)>1]
df_mitigated = df[~df.device_id.isin(to_exclude)].copy()
spill_after = estimate_spillover(df_mitigated)
{"spill_before": spill_before, "spill_after": spill_after, "removed_devices": int(len(to_exclude))}



**Mitigation chosen:** exclude multi‑arm devices + routing fix in prod; add **geo buffers** if spillover persists.  
**Impact:** small effective sample size reduction; improved internal validity.



## Activity 4 — Guardrails
Declare thresholds and produce a PASS/FAIL decision.


In [ ]:

### SOLUTION — Activity 4
d_obs, _ = window_labels(df, exposure_days=14, outcome_days=30)
thresholds = {"max_fpr":0.02, "max_latency_ms":550, "tickets_diff_max":0.001, "chargeback_rate_max":0.007}
guardrails = guardrail_report(d_obs, thresholds)
guardrails



**Decision:** Proceed if **GO**; otherwise pause and analyze the failing guardrail (often latency or FPR).



## Activity 5 — Power, MDE, and Cluster Effects (with CUPED option)
Compute n per arm for a fraud-loss rate MDE; adjust for clustering; try CUPED to see variance reduction.


In [ ]:

### SOLUTION — Activity 5
d_obs, _ = window_labels(df, exposure_days=14, outcome_days=30)
baseline_rate = (d_obs.observed_fraud).mean()
target_mde = 0.0015
n_naive = n_per_arm_for_proportions(baseline_rate, target_mde, alpha=0.05, power=0.8)

cluster_size = 2000; icc = 0.02
deff = design_effect(cluster_size, icc)
n_cluster_adjusted = int(np.ceil(n_naive * deff))

# CUPED demo
pre_proxy = (d_obs.region.map({"NA":0.009,"EU":0.006,"LATAM":0.013,"APAC":0.008}).values
             + np.random.normal(0, 0.001, size=len(d_obs)))
post_rate = d_obs.observed_fraud.astype(float).values
post_adj, theta = cuped_adjust(post_rate, pre_proxy)
var_naive = post_rate.var(ddof=1); var_adj = post_adj.var(ddof=1)

{
 "baseline_rate": float(baseline_rate),
 "n_naive": int(n_naive),
 "design_effect": float(deff),
 "n_cluster_adjusted": n_cluster_adjusted,
 "cuped_theta": float(theta),
 "cuped_variance_reduction_%": float(100*(1 - var_adj/var_naive))
}



## Activity 6 — Staged Rollout & Monitoring/Rollback
Fill the rollout skeleton with phases, entry/exit criteria, and rollback rules.


In [ ]:

### SOLUTION — Activity 6
rollout_plan = {
    "phase_0_canary": {"traffic_share":0.02,"entry":["offline PASS"],"exit":["90% CI excludes 0","SLOs PASS"]},
    "phase_1": {"traffic_share":0.10,"entry":["power on‑track"],"exit":["guardrails PASS 2 weeks"]},
    "phase_2": {"traffic_share":0.50,"entry":["backfill confirms lift"],"exit":["95% CI excludes 0","risk sign‑off"]},
    "phase_3": {"traffic_share":1.00,"entry":["exec GO"],"exit":["post‑launch monitors on"]},
    "rollback": {"trigger":["guardrail FAIL","drift"],"actions":["revert model","disable risky rules","notify on‑call"]}
}
rollout_plan


> End of Solution.